# Full analysis pipeline

Runs the complete two-mark (H3K27me3 / H3K27ac) sSMLM analysis on a folder of cells. These are the same five steps as `python drivers/run_all.py`, run one at a time so you can check each step's QC before going on.

| Step | Script | What it does | Manuscript |
|---|---|---|---|
| 1 | `drivers/01_preprocess.py` | axial filter, nucleus mask (auto threshold), radial contours | Methods, Fig. 3b |
| 2 | `drivers/02_calibrate.py` | inverse Monte Carlo: simulate a matching nucleus, fit DBSCAN `eps(density)` / `min_samples(density)` | Fig. 4 |
| 3 | `drivers/03_cluster.py` | density-corrected DBSCAN per cell and mark, then nanodomain size / count | Figs. 2d/e |
| 4 | `drivers/04_spatial.py` | radial density profiles, SNCR curves and their decay rate | Figs. 3c/f |
| 5 | `drivers/05_compare.py` | pool cells, NT vs. ActD statistics and figures | Fig. 6, Fig. S3 |

**Input:** one folder of RainbowSTORM exports split by mark, `<cell_id>_me3.csv` and `<cell_id>_ac.csv` (columns `x [nm]`, `y [nm]`, `z [nm]`). Conditions come from the cell_id (`CONDITION_PATTERNS` in `drivers/config.py`; by default a cell_id containing `act` is ActD and everything else is NT).

Step 2 is slow, so set `QUICK = True` for a smoke test first. Every step skips cells that already have outputs, so you can re-run any cell of this notebook safely; set `OVERWRITE = True` to redo them.

For one channel of one cell, use `single_channel_nanodomains.ipynb` instead. Open this notebook from the repo root.

In [ ]:
import importlib
import json
import runpy
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

REPO_ROOT = Path.cwd()
DRIVERS = REPO_ROOT / "drivers"
assert DRIVERS.is_dir(), "open this notebook from the repo root"
sys.path.insert(0, str(DRIVERS))
import config as C  # drivers/config.py: every analysis parameter is defined there

## Settings

Paths and run options go here. Analysis parameters (mask threshold, calibration mode, domain scale, DBSCAN floors, SNCR radii, ...) live in [`drivers/config.py`](drivers/config.py). Override any of them by name in `CONFIG_OVERRIDES`; such overrides also take precedence over `QUICK`'s defaults. This cell reloads `config.py` each time it runs, so edits there take effect on the next run.

In [ ]:
DATA_DIR = REPO_ROOT / "raw_data"           # <cell_id>_me3.csv + <cell_id>_ac.csv for every cell
OUTPUT_DIR = REPO_ROOT / "analysis_output"
QUICK = False      # True: small calibration grid, written to <OUTPUT_DIR>_quick (smoke test)
CELLS = None       # None = every cell in DATA_DIR, or a list of cell_ids, e.g. ["k27_k27_thaw009"]
OVERWRITE = False  # False: skip cells whose outputs already exist

CONFIG_OVERRIDES = {
    # "CALIBRATION_MODE": "per_cell",                    # "per_condition" | "per_cell" | "pooled"
    # "DOMAIN_SCALE_NM": {"me3": 200.0, "ac": 250.0},
    # "MASK_THRESH_OVERRIDES": {"some_cell_id": 2.0},    # fix a bad nucleus mask
}

importlib.reload(C)  # start from config.py's values every time
RUN_DIR = Path(OUTPUT_DIR)
if QUICK:
    RUN_DIR = RUN_DIR.with_name(RUN_DIR.name + "_quick")
C.DATA_DIR = Path(DATA_DIR)
C.OUTPUT_DIR = RUN_DIR
C.QUICK_OVERRIDES["OUTPUT_DIR"] = RUN_DIR
for name, value in CONFIG_OVERRIDES.items():
    if not hasattr(C, name):
        raise KeyError(f"drivers/config.py has no setting {name!r}")
    setattr(C, name, value)
    C.QUICK_OVERRIDES.pop(name, None)

print(f"data:   {C.DATA_DIR}")
print(f"output: {RUN_DIR}")
print(f"calibration mode {C.CALIBRATION_MODE} | use_z {C.USE_Z} | domain scale {C.DOMAIN_SCALE_NM} nm | "
      f"seed placement {C.SEED_PLACEMENT}")

In [ ]:
def run_step(script):
    """Run one drivers/ step in this kernel with the settings above (same as python drivers/<script> <flags>)."""
    argv = [script]
    if QUICK:
        argv.append("--quick")
    if CELLS:
        argv += ["--cells", *CELLS]
    if OVERWRITE:
        argv.append("--overwrite")
    saved_argv = sys.argv
    sys.argv = argv
    try:
        runpy.run_path(str(DRIVERS / script), run_name="__main__")
    finally:
        sys.argv = saved_argv


def show(*paths, width=700):
    """Display figures the steps saved (the drivers plot headless and write every figure to disk)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(RUN_DIR)}`"))
            display(Image(filename=str(p), width=width))
        else:
            print(f"not found: {p}")


def cell_table(filename):
    """One row per cell from the cells/<cell_id>/<filename> JSON summaries; per-mark fields become me3.* / ac.*."""
    rows = [pd.json_normalize(json.loads(f.read_text())) for f in sorted((RUN_DIR / "cells").glob(f"*/{filename}"))]
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

## 1. Preprocess

Axial filter, nucleus mask, and 100 radial contour bands per cell. **Check the mask covers the whole nucleus.** If it doesn't, add the cell to `MASK_THRESH_OVERRIDES` in `CONFIG_OVERRIDES`, then re-run with `OVERWRITE = True`. QC figures are shown for `PREVIEW`; set it to any cell_id to inspect another cell.

In [ ]:
run_step("01_preprocess.py")

In [ ]:
manifest = pd.read_csv(RUN_DIR / "manifest.csv")
PREVIEW = manifest["cell_id"].iloc[0] if not CELLS else CELLS[0]

display(cell_table("preprocess.json").round(2))
show(RUN_DIR / "cells" / PREVIEW / "qc_mask.png", RUN_DIR / "cells" / PREVIEW / "qc_contours.png")

## 2. Calibrate DBSCAN (slow)

For each calibration group (`CALIBRATION_MODE`: per condition on its densest cell, per cell, or pooled), this simulates a nucleus matching the real one with known nanodomains. It then thins the simulation to a range of densities and fits `eps(density)` and `min_samples(density)` so DBSCAN recovers the true domain size at every density.

**Check:** the simulated g(r) and L(r) should overlay the real ones. In the size-vs-density plot, corrected sizes should stay flat while fixed parameters drift (Fig. 4b vs 4c).

In [ ]:
run_step("02_calibrate.py")

In [ ]:
for cal_dir in sorted((RUN_DIR / "calibration").iterdir()):
    for ch in C.CHANNELS:
        cal = json.loads((cal_dir / f"{ch}_calibration.json").read_text())
        print(f"[{cal_dir.name}] {C.CHANNEL_LABELS[ch]}: eps = {cal['eps_coef']:.3g} * density^{cal['eps_exp']:.3f} "
              f"(R^2 {cal['eps_r_squared']:.2f}) | min_samples = {cal['min_samples_slope']:.4g} * density + "
              f"{cal['min_samples_intercept']:.2f} | ground-truth diameter {cal['gt_target']:.0f} nm | "
              f"converged: {cal['converged']}")
        show(cal_dir / f"{ch}_pair_correlation.png", cal_dir / f"{ch}_size_vs_density.png", width=600)

## 3. Density-corrected DBSCAN

Each cell and mark is clustered with `eps` / `min_samples` read off its calibration at that cell's own density, and nanodomain size and count are measured. `extrapolated = True` means the cell's density lies outside the calibrated range.

In [ ]:
run_step("03_cluster.py")

In [ ]:
clusters = cell_table("cluster.json")
cols = ["cell_id", "condition", "calibration"] + [
    f"{ch}.{k}" for ch in C.CHANNELS for k in ("density_per_um2", "eps", "min_samples", "n_nanodomains",
                                                "mean_bb_size_nm", "frac_locs_clustered", "extrapolated")]
clusters[[c for c in cols if c in clusters.columns]].round(2)

## 4. Radial profiles and SNCR

Per cell: each mark's density across the radial contour bands, plus the self:nonself contact ratio (SNCR) around every nanodomain with its exponential decay rate. The figure is for `PREVIEW`.

In [ ]:
run_step("04_spatial.py")
show(RUN_DIR / "cells" / PREVIEW / "spatial.png")

## 5. Pool cells and compare conditions

Writes the pooled tables and condition figures to `results/`; `stats.csv` holds the Mann-Whitney tests against the reference condition.

In [ ]:
run_step("05_compare.py")

In [ ]:
results = RUN_DIR / "results"
display(pd.read_csv(results / "stats.csv").round(4))
show(*(results / f for f in ("fig6g_nanodomain_size.png", "fig6h_nanodomain_count.png",
                             "fig6_radial_by_condition.png", "fig6_sncr_by_condition.png",
                             "figS3_ratio_and_decay.png")))

### Outputs

Everything is under `RUN_DIR`. The full layout is described at the top of `drivers/common.py`:

- `results/per_cell_summary.csv`: one row per cell, with every headline number
- `results/nanodomains_all.csv`, `sncr_all.csv`, `sncr_fits_all.csv`, `radial_profiles_all.csv`: pooled per-nanodomain / per-curve tables
- `cells/<cell_id>/`: masked localizations, mask, clustered localizations, nanodomain tables, QC figures
- `calibration/<key>/`: each calibration's JSON, simulated nucleus and diagnostic figures

In [ ]:
pd.read_csv(results / "per_cell_summary.csv")